In [ ]:
import xarray as xr
import numpy as np
from workflow.scripts.utils import make_consistent
from dask.diagnostics import ProgressBar
from dask.distributed import Client, LocalCluster

In [ ]:
tas = xr.open_dataset(snakemake.input.tas[0])
ps = xr.open_dataset(snakemake.input.pa[0])
tauv = xr.open_dataset(snakemake.input.tauv[0])
tauu = xr.open_dataset(snakemake.input.tauu[0])

In [ ]:
if snakemake.wildcards.model == 'IPSL-CM6A-LR-INCA' or snakemake.wildcards.model == 'CNRM-ESM2-1':
    tas = tas.rename_dims(bounds='bnds').rename({'lat_bounds':'lat_bnds','lon_bounds':'lon_bnds'})
    ps = ps.rename_dims(bounds='bnds').rename({'lat_bounds':'lat_bnds','lon_bounds':'lon_bnds'})
    tauv = tauv.rename_dims(bounds='bnds').rename({'lat_bounds':'lat_bnds','lon_bounds':'lon_bnds'})
    tauu = tauu.rename_dims(bounds='bnds').rename({'lat_bounds':'lat_bnds','lon_bounds':'lon_bnds'})

In [ ]:
tauu, tauv, tas, ps = make_consistent([tauv, tauu, tas, ps])

stress = xr.merge([tauu, tauv, tas, ps])

In [ ]:
if snakemake.config.get('dask',False):
    cluster = LocalCluster(n_workers=2,threads_per_worker=2,memory_limit='8G')
    
    client = Client(cluster)
    stress = stress.chunk({'time':12})

In [ ]:
def calc_surf_air_density(ps,tas):
    " density  = pressure / Rair / ta[ilev,:,:]  "
    Rair  = 287.058
    rho_a = (ps / Rair)/tas
    return rho_a

def calc_surf_stress(tauv, tauu):
    func = lambda tv, tu: np.sqrt(tv**2+tu**2)
    return xr.apply_ufunc(func,tauv,tauu, dask='allowed')

In [ ]:

rho_a = calc_surf_air_density(stress['ps'], stress['tas']).compute()
tau_srf = calc_surf_stress(stress['tauv'], stress['tauu']).compute()

In [ ]:
ustar = np.sqrt(tau_srf/rho_a)
ustar = ustar.resample(time='Y').mean()
ustar.attrs['units']='m s-1'
ustar.attrs['long_name']='Friction velocity'


In [ ]:
out_ds = ustar.to_dataset(name='ustar')

In [ ]:
out_ds.attrs = tauv.attrs.copy()
out_ds.attrs['variable_id']='ustar'
out_ds.attrs.pop('intake_esm_attrs:variable_id')
out_ds.attrs.pop('intake_esm_vars')

In [ ]:

out_ds.to_netcdf(snakemake.output.outpath)